In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gc
import evaluate
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer
from Data_Loader import Sen1Floods11Dataset, training_augmentation
from peft import LoraConfig, get_peft_model

In [ ]:
sen1floods11_data_folder = "../Data/Sen1Floods11/"
splits = sen1floods11_data_folder + "splits/"
SAR_data  = sen1floods11_data_folder + "S1Hand/"
Label_data = sen1floods11_data_folder + "LabelHand/"

In [ ]:
train_files = pd.read_csv(splits + 'flood_train_data.csv',header=None, names=["SAR", "Label"])
val_files = pd.read_csv(splits + 'flood_valid_data.csv',header=None, names=["SAR", "Label"])
test_files = pd.read_csv(splits + 'flood_test_data.csv',header=None, names=["SAR", "Label"])

In [ ]:
train_dataset = Sen1Floods11Dataset(SAR_data, Label_data, train_files, transform=training_augmentation)
val_dataset = Sen1Floods11Dataset(SAR_data, Label_data, val_files)
test_dataset = Sen1Floods11Dataset(SAR_data, Label_data, test_files)

### Custom Loss Functions

In [ ]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]

        upsampled_logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        #Upsampling the segformer output to match the true mask pixel-by-pixel

        class_weights = torch.tensor([1.0, 5.0]).to(self.args.device)
        #The relative weightage to be given to pixels of each class

        loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
        loss = loss_fn(upsampled_logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
class SARHybridLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]

        upsampled_logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)

        probs = torch.softmax(upsampled_logits, dim=1)
        valid_mask = (labels != 255).float()

        true_land = (labels == 0).float() * valid_mask
        true_water = (labels == 1).float() * valid_mask

        pred_land = probs[:, 0, :, :] * valid_mask
        pred_water = probs[:, 1, :, :] * valid_mask

        # Multi-Class Focal Loss
        gamma = 2.0 #Higher gamma downweights the common(land) class
        eps = 1e-6

        focal_land = -true_land * ((1.0 - pred_land) ** gamma) * torch.log(pred_land + eps)
        focal_water = -true_water * ((1.0 - pred_water) ** gamma) * torch.log(pred_water + eps)
        focal_loss = (focal_land + focal_water).sum() / (valid_mask.sum() + eps)

        #Dice Loss
        intersection = torch.sum(pred_water * true_water, dim=(-2, -1))
        denominator = torch.sum(pred_water, dim=(-2, -1)) + torch.sum(true_water, dim=(-2, -1))

        dice_coeff = (2.0 * intersection + eps) / (denominator + eps)
        dice_loss = 1.0 - torch.mean(dice_coeff)


        total_loss = (0.5 * focal_loss) + (0.5 * dice_loss)

        return (total_loss, outputs) if return_outputs else total_loss

### Training Setup

In [ ]:
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred

        #Converting the model output logits to the class predictions
        logits_tensor = torch.from_numpy(logits)
        upsampled_logits = torch.nn.functional.interpolate(
            logits_tensor, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        predictions = upsampled_logits.argmax(dim=1).numpy()

        metrics = metric.compute(
            predictions=predictions,
            references=labels,
            num_labels=2,
            ignore_index=255,
            reduce_labels=False)

        per_category_iou = metrics["per_category_iou"]
        return {
            "mIoU": metrics["mean_iou"],
            "pixel_accuracy": metrics["mean_accuracy"],
            "land_iou": per_category_iou[0],
            "flood_water_iou": per_category_iou[1]}

In [ ]:
model_id = "nvidia/segformer-b2-finetuned-ade-512-512"

id2label = {0: "land", 1: "flood_water"}
label2id = {"land": 0, "flood_water": 1}

### Model Runs

In [ ]:
#Using normal cross entropy loss function

model_base = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)


training_args = TrainingArguments(
    output_dir="./segformer_results",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none")


trainer = Trainer(
    model=model_base,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)


trainer.train()

model_base.save_pretrained("../Models/Cross_entropy")

del model_base
del trainer
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  110MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  110MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Miou,Pixel Accuracy,Land Iou,Flood Water Iou
1,0.582116,0.531021,0.633875,0.789048,0.876771,0.390979
2,0.470398,0.388672,0.692777,0.760517,0.925269,0.460285
3,0.403389,0.330494,0.743414,0.828579,0.933754,0.553075
4,0.365873,0.305368,0.746772,0.819514,0.937152,0.556392
5,0.349209,0.294720,0.746502,0.810772,0.938827,0.554178


/usr/local/lib/python3.13/dist-packages/datasets/features/image.py:406: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Using weighted cross entropy loss function

model_base = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)

training_args = TrainingArguments(
    output_dir="./segformer_results",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none")


trainer = WeightedLossTrainer(
    model=model_base,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)


trainer.train()

model_base.save_pretrained("../Models/Weighted_Cross_entropy")

del model_base
del trainer
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([2, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Miou,Pixel Accuracy,Land Iou,Flood Water Iou
1,0.565264,0.475402,0.607206,0.848761,0.832673,0.381738
2,0.477217,0.378757,0.707807,0.876496,0.903879,0.511734
3,0.427603,0.359375,0.737023,0.878088,0.920435,0.553610
4,0.397964,0.312965,0.732433,0.897408,0.913983,0.550883
5,0.390720,0.325732,0.735530,0.899619,0.915291,0.555770


/usr/local/lib/python3.13/dist-packages/datasets/features/image.py:406: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Using FOCAL+DICE loss function

model_base = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)

training_args = TrainingArguments(
    output_dir="./segformer_results",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none")


trainer = SARHybridLossTrainer(
    model=model_base,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)


trainer.train()

model_base.save_pretrained("../Models/Focal_Dice")

del model_base
del trainer
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([2, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Miou,Pixel Accuracy,Land Iou,Flood Water Iou
1,0.490413,0.445856,0.702464,0.831578,0.911475,0.493454
2,0.461446,0.417473,0.747327,0.858156,0.929692,0.564962
3,0.443441,0.406634,0.768996,0.856393,0.940026,0.597967
4,0.447777,0.397565,0.778181,0.857997,0.943664,0.612699
5,0.448909,0.396335,0.775843,0.866678,0.941244,0.610441


/usr/local/lib/python3.13/dist-packages/datasets/features/image.py:406: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# FOCAL+DICE + DoRA

model_base = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)

dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
    use_dora=True,
    modules_to_save=["decode_head"]
)

model_dora = get_peft_model(model_base, dora_config)

training_args = TrainingArguments(
    output_dir="./segformer_results",
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none")


trainer = SARHybridLossTrainer(
    model=model_dora,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)


trainer.train()

model_dora.save_pretrained("../Models/DoRA_16")

del model_base
del model_dora
del trainer
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([2, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Miou,Pixel Accuracy,Land Iou,Flood Water Iou
1,0.485067,0.458530,0.531264,0.586314,0.895090,0.167438
2,0.469173,0.440697,0.623582,0.686173,0.910645,0.336518
3,0.453496,0.426624,0.694198,0.770495,0.923142,0.465253
4,0.449554,0.418046,0.718294,0.792557,0.929777,0.506812
5,0.455828,0.413977,0.727751,0.808997,0.930525,0.524976


/usr/local/lib/python3.13/dist-packages/datasets/features/image.py:406: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


In [ ]:
# FOCAL+DICE + DoRA (Rank 32)

model_base = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True)

dora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
    use_dora=True,
    modules_to_save=["decode_head"]
)

model_dora = get_peft_model(model_base, dora_config)

training_args = TrainingArguments(
    output_dir="./segformer_results",
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none")


trainer = SARHybridLossTrainer(
    model=model_dora,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics)


trainer.train()

model_dora.save_pretrained("../Models/DoRA_32")

del model_base
del model_dora
del trainer
gc.collect()
torch.cuda.empty_cache()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([2, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Miou,Pixel Accuracy,Land Iou,Flood Water Iou
1,0.483063,0.453318,0.568916,0.625184,0.901961,0.235871
2,0.462871,0.430735,0.690611,0.765594,0.922605,0.458616
3,0.444714,0.411649,0.741301,0.834498,0.931530,0.551073
4,0.439451,0.402134,0.752368,0.834127,0.936757,0.567980
5,0.445334,0.397072,0.751999,0.840281,0.935392,0.568606


/usr/local/lib/python3.13/dist-packages/datasets/features/image.py:406: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")
